# Matrix Factorization — bản TỐI GIẢN để hiểu BẢN CHẤT

Không PyTorch, không GPU, không autograd. Chỉ NumPy + gradient tính bằng tay,
chạy trên một ma trận tí hon để bạn **NHÌN THẤY** nó hoạt động.

## Ý TƯỞNG CỐT LÕI

Ta có ma trận đánh giá $R$ cỡ $(n\_users \times n\_items)$. Phần lớn ô bị **TRỐNG**
(user chưa xem tin đó). Câu hỏi: đoán các ô trống đó là bao nhiêu?

Giả thiết của MF: mỗi user và mỗi item có thể mô tả bằng $k$ "yếu tố ẩn"
(latent factors). Ví dụ với phim: $k=2$ có thể là [mức "hành động", mức "lãng mạn"].

- user $u$  ->  vector $p_u$  (gu thích từng yếu tố tới đâu)
- item $i$  ->  vector $q_i$  (item chứa từng yếu tố tới đâu)

Dự đoán = mức khớp giữa hai vector = **TÍCH VÔ HƯỚNG**:

$$R[u, i] \approx p_u \cdot q_i$$

Viết gọn cho cả ma trận: $R \approx P \, Q^T$

- $P$: $(n\_users \times k)$ — mỗi DÒNG là 1 user
- $Q$: $(n\_items \times k)$ — mỗi DÒNG là 1 item

"Factorization" = tách $R$ (to, nhiều ô trống) thành tích của hai ma trận GẦY
($P$ và $Q$, ít cột $k$). Vì $k$ nhỏ, mô hình buộc phải tìm ra cấu trúc chung -> nó
tự điền được các ô trống một cách hợp lý.

## HỌC P, Q NHƯ THẾ NÀO?

Chỉ so khớp trên các ô **ĐÃ BIẾT**. Sai số tại ô $(u, i)$:

$$e = R[u, i] - p_u \cdot q_i$$

Hàm mất mát (chỉ trên ô đã biết) + phạt L2 để chống overfit:

$$L = \sum e^2 + \lambda \left( \|p_u\|^2 + \|q_i\|^2 \right)$$

Đạo hàm theo $p_u$ và $q_i$ (đây là toàn bộ "phép màu", tự tính tay):

$$\frac{\partial L}{\partial p_u} = -2 e\, q_i + 2\lambda p_u$$
$$\frac{\partial L}{\partial q_i} = -2 e\, p_u + 2\lambda q_i$$

Gradient descent: đi NGƯỢC hướng đạo hàm một bước nhỏ (lr). Bỏ hằng số 2 vào lr:

$$p_u \leftarrow p_u + lr \, (e\, q_i - \lambda p_u)$$
$$q_i \leftarrow q_i + lr \, (e\, p_u - \lambda q_i)$$

Lặp lại trên mọi ô đã biết, nhiều vòng (epoch), là xong. Chỉ vậy thôi.

In [1]:
import numpy as np

## Hàm matrix_factorization

In [2]:
def matrix_factorization(R, mask, k=2, lr=0.01, reg=0.1, epochs=5000, seed=0):
    """
    R    : ma trận đánh giá (n_users × n_items). Ô trống điền số bất kỳ (sẽ bị mask bỏ qua).
    mask : ma trận 0/1 cùng cỡ. mask[u, i] = 1 nghĩa là ô đó ĐÃ BIẾT (đưa vào học).
    k    : số yếu tố ẩn.
    lr   : learning rate (độ dài mỗi bước cập nhật).
    reg  : λ — hệ số phạt L2.

    Trả về P, Q sao cho  P @ Q.T  xấp xỉ R trên các ô đã biết.
    """
    rng = np.random.default_rng(seed)
    n_users, n_items = R.shape

    # Khởi tạo ngẫu nhiên nhỏ. Ban đầu P, Q vô nghĩa; gradient descent sẽ nắn dần.
    P = rng.normal(0, 0.1, size=(n_users, k))
    Q = rng.normal(0, 0.1, size=(n_items, k))

    for epoch in range(epochs):
        # Duyệt qua TỪNG ô đã biết và cập nhật ngay (SGD).
        for u in range(n_users):
            for i in range(n_items):
                if mask[u, i] == 0:
                    continue  # ô trống -> không học

                # Sai số dự đoán tại ô này.
                pred = P[u] @ Q[i]
                e = R[u, i] - pred

                # Cập nhật đồng thời (dùng P[u] cũ để tính Q[i] cho đúng).
                p_u_old = P[u].copy()
                P[u] += lr * (e * Q[i]     - reg * P[u])
                Q[i] += lr * (e * p_u_old  - reg * Q[i])

        # Thỉnh thoảng in lỗi RMSE trên các ô đã biết để theo dõi hội tụ.
        if epoch % 1000 == 0 or epoch == epochs - 1:
            pred_full = P @ Q.T
            err = (R - pred_full) * mask          # chỉ tính ô đã biết
            rmse = np.sqrt((err ** 2).sum() / mask.sum())
            print(f"epoch {epoch:>5}  RMSE = {rmse:.4f}")

    return P, Q

## Demo: dữ liệu user × item

Số = mức thích (1..5). 0 = CHƯA xem (ô trống cần đoán).

In [3]:
#            item:  i0  i1  i2  i3
R = np.array([
    [5, 3, 0, 1],   # user 0
    [4, 0, 0, 1],   # user 1
    [1, 1, 0, 5],   # user 2
    [1, 0, 0, 4],   # user 3
    [0, 1, 5, 4],   # user 4
], dtype=float)

mask = (R > 0).astype(float)  # 1 ở ô đã biết, 0 ở ô trống

print("Ma trận gốc (0 = chưa biết):")
print(R)

Ma trận gốc (0 = chưa biết):
[[5. 3. 0. 1.]
 [4. 0. 0. 1.]
 [1. 1. 0. 5.]
 [1. 0. 0. 4.]
 [0. 1. 5. 4.]]


## Huấn luyện

In [4]:
print("Đang học P, Q ...")
P, Q = matrix_factorization(R, mask, k=2, lr=0.01, reg=0.1, epochs=5000)

Đang học P, Q ...
epoch     0  RMSE = 3.2553
epoch  1000  RMSE = 0.1373
epoch  2000  RMSE = 0.1372
epoch  3000  RMSE = 0.1372
epoch  4000  RMSE = 0.1372
epoch  4999  RMSE = 0.1372


## Kết quả: tái tạo ma trận và xem vector ẩn

In [5]:
R_hat = P @ Q.T
np.set_printoptions(precision=2, suppress=True)

print("Ma trận tái tạo (P @ Q.T) — các ô 0 giờ đã được ĐOÁN:")
print(R_hat)

print("\nVector ẩn của từng user (P):")
print(P)
print("\nVector ẩn của từng item (Q):")
print(Q)

Ma trận tái tạo (P @ Q.T) — các ô 0 giờ đã được ĐOÁN:
[[4.77 2.84 1.73 1.02]
 [3.82 2.29 1.61 0.99]
 [1.01 0.98 5.8  4.72]
 [0.99 0.9  4.74 3.84]
 [1.14 0.99 4.86 3.93]]

Vector ẩn của từng user (P):
[[-1.99  1.1 ]
 [-1.63  0.8 ]
 [-1.43 -1.65]
 [-1.23 -1.29]
 [-1.3  -1.28]]

Vector ẩn của từng item (Q):
[[-1.85  1.  ]
 [-1.19  0.44]
 [-1.9  -1.86]
 [-1.41 -1.63]]
